In [8]:
from typing import TypedDict, Annotated, List, Literal, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import START, END, MessagesState
from langgraph.graph.state import StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model


import random
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
# class SupervisorState(MessagesState):
#     """State for supervising multi-agent workflow execution."""
#     current_agent: str = ""
#     task_assignments: Dict[str, Any] = {}
#     agent_outputs: Dict[str, Any] = {}
#     workflow_stage: str = "initial"
#     iteration_count: int = 0
#     max_iterations: int = 10
#     final_output: str = ""


class SupervisorState(MessagesState):
    """State for supervising multi-agent workflow execution."""
    next_agent: str = ""
    research_data: str = ""
    analysis: str = ""
    final_report: str = ""
    task_complete: bool = False
    current_task: str = ""
    


In [9]:
llm = init_chat_model("gpt-4o")

def create_supervisor_chain():
    """Creates a supervisor chain that oversees the multi-agent workflow."""

    supervisor_prompt = ChatPromptTemplate.from_messages([
        (
            "system", 
            """
            
                You are a supervisor agent overseeing a multi-agent workflow. Your job is to monitor the agents, evaluate their outputs, and provide feedback to ensure the successful completion of the overall task.

                1. Researcher Agent: Responsible for gathering information and data relevant to the task.
                2. Analyst Agent: Responsible for analyzing the research data and providing insights.
                3. Writer Agent: Responsible for compiling the research and analysis into a final report.        

                Based on the current state and conversation, determine which agent should be active next, what feedback to provide to the agents, and when the task is complete. Your goal is to ensure that the final output is of high quality and meets the requirements of the task.
                If the task is complte, respond with 'DONE'.

                Current state:
                - Has research data: {has_research_data}
                - Has analysis: {has_analysis}
                - Has report: {has_report}

                Respond with only the agent name that should be active next (Researcher, Analyst, Writer) or 'DONE' if the task is complete.

            """
        ),
        ("human", "{task}")
    ])

    return supervisor_prompt | llm

In [ ]:
def supervisor_agent(state: SupervisorState) -> Dict:
    """Supervisor agent that decides which agent should be active next based on the current state."""
    
    messages = state["messages"]
    task = messages[-1].content if messages else "No task provided."

    has_research = bool(state.research_data)
    has_analysis = bool(state.analysis)
    has_report = bool(state.final_report)

    chain = create_supervisor_chain()
    decision = chain.invoke(task=task, has_research_data=has_research, has_analysis=has_analysis, has_report=has_report)